In [4]:
# Library imports
import torch
import math
import random
import json
import nltk
import itertools
import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity
from external.claimify.src.claimify import Claimify

In [2]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA RTX 500 Ada Generation Laptop GPU


In [3]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 0.0 MB; Reserved: 0.0 MB


In [4]:
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

CLAIM_MODEL = "Qwen/Qwen3.5-0.8B"
CLAIM_MODEL_DTYPE = torch.float16

NLTK_MODEL = 'tokenizers/punkt/english.pickle'

In [5]:
#nltk.download('punkt_tab')

In [6]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

claim_tokenizer = AutoTokenizer.from_pretrained(CLAIM_MODEL)
claim_model = AutoModelForCausalLM.from_pretrained(CLAIM_MODEL, dtype=CLAIM_MODEL_DTYPE).to(device)

embed_model = SentenceTransformer(EMBED_MODEL).to(device)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

# Label examples
for example in dataset['pos'].values():
	example['label'] = 1.0 

for example in dataset['neg'].values():
	example['label'] = 0.0 

# Flatten list
all_examples = [ *dataset['pos'].values(), *dataset['neg'].values() ]

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [11]:

def get_closest_sentence_pairs(sentences, embeddings, top_k=None):
    """
    Given a list of sentences and their embeddings, return sentence pairs with closest embeddings.
    
    Args:
        sentences: list of sentence strings
        embeddings: numpy array of embeddings for each sentence
        top_k: number of closest pairs to return
    
    Returns:
        list of tuples (sentence_a, sentence_b, similarity_score)
    """
    
    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(embeddings)
    
    for i in range(len(sentences)):
        similarity_matrix[i][i] = -1.0  # Exclude self-similarity by setting diagonal to -1

    pairs = [ (i, np.argmax(similarity_matrix[i])) for i in range(len(sentences)) ]
    
    # Sort by similarity score (descending) and get top-k
    if top_k is not None:
        pairs.sort(key=lambda x: similarity_matrix[x[0]][x[1]], reverse=True)
        pairs = pairs[:top_k]
        
    # Return pairs with sentences and similarity scores
    return [(sentences[i], sentences[j], similarity_matrix[i][j]) for i, j in pairs]

def predict(premise, hypothesis):
    input = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = nli_model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}
    

def claim_extractor(prompt, temperature):
    # prepare the model input
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = claim_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = claim_tokenizer([text], return_tensors="pt").to(device)

    # conduct text completion
    generated_ids = claim_model.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    #thinking_content = claim_tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = claim_tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print(f'LLM:')
    print(f'Prompt:\n{prompt}\n')
    print(f'Response:\n{content}\n')

    return content

def extract_claims(question, answer):
    c = Claimify(llm_function=claim_extractor)
    claims = c.extract_claims(question=question, answer=answer)
    print(claims)

def detect(document):
    sentences = nltk.tokenize.sent_tokenize(document, language='english')
    sentence_count = len(sentences)

    print(f'Split text into {sentence_count} sentences.')

    prompt = 'Extract all facts from the following text. Respond with only the facts, with one fact per line. Each fact should be self-contained. Remove all pronouns and replace with proper nouns of the entity they refer to.'
    claims = claim_extractor(f'{prompt}\n\nText:\n{document}', 1.0).split('\n')

    print(f'Extracted {len(claims)} claims:')
    print(claims)

    embeddings = embed_model.encode(claims)
    pairs = get_closest_sentence_pairs(claims, embeddings)

    p_contra_max = 0.0
    evidence = []

    for a, b, similarity in tqdm.tqdm(pairs):
        prediction = predict(a, b)
        p_contra = prediction['contradiction']
        p_contra_max = max(p_contra_max, p_contra)

        print(f'Comparing:')
        print(f'  {a}')
        print(f'  {b}')
        print(f'  Similarity: {similarity:.4f}')
        print(f'  Prediction: {p_contra:.4f}')
        print()

        if p_contra > 0.75:
            evidence.append((a, b, p_contra))

    return p_contra_max, evidence

In [9]:
premise = "The sky is red"
hypothesis = "The sky is blue"

print(predict(premise, hypothesis))

{'entailment': 8.13603401184082e-05, 'neutral': 0.0005097389221191406, 'contradiction': 0.99951171875}


In [13]:
pos = False					# Test example from positive or negative set
example_id = None			# Specific example ID to test, or None for random

# 3489738232_6
# story_train_3585

examples = dataset['pos' if pos else 'neg']

if example_id is not None:
	example = examples[example_id]
else:
	examples = list(examples.values())
	example = examples[random.randint(0, len(examples) - 1)]

print(f'Detecting contradictions in example {example["unique id"]}')
print()

print(example['text'])
print()

if pos:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence = detect(example['text'])
p_contra, evidence

Detecting contradictions in example wiki_train_29223

 = HMS Resolution ( 1892 ) = 
 
 HMS Resolution was a Royal Sovereign - class pre - dreadnought battleship of the Royal Navy. The ship was built by Palmers Shipbuilding and Iron Company , starting with her keel laying in June 1890. She was launched in May 1892 and , after completing trials , was commissioned into the Channel Squadron the following December. She was armed with a main battery of four 13. 5 - inch guns and a secondary battery of ten 6 - inch guns. The ship had a top speed of 16. 5 knots. 
 Resolution served with the Channel Squadron up to 1901 ; she took part in the Diamond Jubilee Fleet Review and a number of manoeuvres in the Atlantic and the Southwest Approaches. She was recommissioned as a coast guard ship later in 1901 and underwent a refit in 1903 , after which she served at Sheerness as a port guard ship , before entering the Fleet Reserve at Chatham in June 1904. She suffered damage while participating in combi

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


LLM:
Prompt:
Extract all facts from the following text. Respond with only the facts, with one fact per line. Each fact should be self-contained. Remove all pronouns and replace with proper nouns of the entity they refer to.

Text:
 = HMS Resolution ( 1892 ) = 
 
 HMS Resolution was a Royal Sovereign - class pre - dreadnought battleship of the Royal Navy. The ship was built by Palmers Shipbuilding and Iron Company , starting with her keel laying in June 1890. She was launched in May 1892 and , after completing trials , was commissioned into the Channel Squadron the following December. She was armed with a main battery of four 13. 5 - inch guns and a secondary battery of ten 6 - inch guns. The ship had a top speed of 16. 5 knots. 
 Resolution served with the Channel Squadron up to 1901 ; she took part in the Diamond Jubilee Fleet Review and a number of manoeuvres in the Atlantic and the Southwest Approaches. She was recommissioned as a coast guard ship later in 1901 and underwent a refit

  5%|▌         | 3/56 [00:00<00:02, 22.60it/s]

Comparing:
  - HMS Resolution was a Royal Sovereign class pre-dreadnought battleship of the Royal Navy.
  - The Royal Sovereign-class battleships were based on Admiral-class barbette ships, but contained several alterations.
  Similarity: 0.6617
  Prediction: 0.0002

Comparing:
  - The ship was built by Palmers Shipbuilding and Iron Company, starting with her keel laying in June 1890.
  - She was laid down on 14 June 1890, launched on 28 May 1892 and completed the following November.
  Similarity: 0.6486
  Prediction: 0.0003

Comparing:
  - She was launched in May 1892 and commissioned into the Channel Squadron the following December.
  - She underwent trials in December 1893, and was commissioned at Portsmouth on 5 December of that year for service in the Channel Squadron.
  Similarity: 0.8018
  Prediction: 0.0007

Comparing:
  - She was armed with a main battery of four 13.5-inch guns and a secondary battery of ten 6-inch guns.
  - She displaced up to 15,580 tonnes at her full combat

...

 95%|█████████▍| 53/56 [00:02<00:00, 17.41it/s]

Comparing:
  - In the summer of 1906, she took part in manoeuvres during which she suffered slight damage when she collided with her sister ship HMS Ramillies near the Tongue Lightship on 15 July 1906.
  - On 18 July 1896, she collided with her sister ship HMS Repulse, suffering slight plating and keel damage.
  Similarity: 0.7555
  Prediction: 0.9868

Comparing:
  - Later that year, she underwent another refit at Chatham.
  - She underwent a refit in 1903, after which she served at Sheerness as a port guard ship.
  Similarity: 0.6901
  Prediction: 0.0003

Comparing:
  - On 12 February 1907, Resolution transferred to the Special Service Division of the Home Fleet at Devonport.
  - She was recommissioned into the Special Service Division of the Home Fleet the following year.
  Similarity: 0.5847
  Prediction: 0.0010

Comparing:
  - She remained in that service until 8 August 1911.
  - She was decommissioned in August 1911 and laid up at Motherbank for disposal.
  Similarity: 0.6636
  Pr

100%|██████████| 56/56 [00:03<00:00, 17.90it/s]

Comparing:
  - She was then laid up at the Motherbank, awaiting disposal.
  - She was decommissioned in August 1911 and laid up at Motherbank for disposal.
  Similarity: 0.7500
  Prediction: 0.0001

Comparing:
  - On 2 April 1914, Resolution was sold as scrap for £35,650 to F. Rijsdijk; the following month, she was towed to the Netherlands to be broken up.
  - She was sold for scrap in April 1914 and towed to the Netherlands to be broken up the following month.
  Similarity: 0.7543
  Prediction: 0.0003



(0.99853515625,
 [('- She underwent a refit in 1903, after which she served at Sheerness as a port guard ship.',
   '- She was recommissioned as a coast guard ship later in 1901.',
   0.99169921875),
  ('- Resolution also carried ten quick-fire (QF) 6-inch guns, four of which were mounted in casemates on the main deck, plus sixteen QF 6-pounder Hotchkiss guns and twelve QF 3-pounder Hotchkiss guns.',
   '- Resolution was armed with four breech-loading 13.5-inch guns on two barbettes with armour ranging from 11 to 17 inches in thickness.',
   0.99853515625),
  ('- She was recommissioned for further Channel Fleet service on 9 April 1895.',
   '- She was launched in May 1892 and commissioned into the Channel Squadron the following December.',
   0.8466796875),
  ('- She nonetheless took part in annual manoeuvres from 24 July 1896 to 30 July 1896, this time off the south-west coasts of England and Ireland as part of "Fleet A".',
   '- From 29 July 1899 to 4 August 1899, she participated in

In [17]:
# Evaluation loop

results = []
for example in tqdm.tqdm(all_examples):
	unique_id = example["unique id"]
	text = example["text"]
	y_true = example["label"]        # 0 or 1
	doc_type = example["doc_type"]   # story/wiki/news

	p_pred, evidence = (random.random(), []) #predict(text)

	results.append({
		"unique_id": unique_id,
		"p_pred": p_pred,
		"y_true": y_true,
		"doc_type": doc_type
	})

100%|██████████| 891/891 [00:00<00:00, 68543.43it/s]


In [14]:
# Write output file
with open('./data/results.json', "w") as f:
    json.dump(results, f, indent=2)